In [ ]:
import os 
# Get the current working directory
current_dir = os.getcwd()
print("Current working directory:", current_dir)

import pandas as pd
import scanpy as sc
import anndata as ad
from tqdm import tqdm
import matplotlib.pyplot as plt # import matplotlib to visualize our qc metrics

# magic incantation to help matplotlib work with our jupyter notebook
%matplotlib inline 

sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')

In [ ]:
import sys
sys.path.append('../../utils')
from functions import * 

In [ ]:
# load adata object for tabula muris 
adata = sc.read_h5ad("/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaMurisBrain/adata.h5ad")

### Let's do some quick QC of the raw counts from featureCounts

In [ ]:
# Remove genes from adata that have "ERCC" in their name, these are just spike ins 
adata = adata[:, [not x.startswith('ERCC') for x in adata.var_names]]
# Also remove transgenes, any gene name that ends in "_transgene"
adata = adata[:, [not x.endswith('_transgene') for x in adata.var_names]]
# Also remove "LOC" genes for now, any gene name that starts with "LOC"
adata = adata[:, [not x.startswith('LOC') for x in adata.var_names]]
# Also remove recombinant protiens, gene names that start with MGC
adata = adata[:, [not x.startswith('MGC') for x in adata.var_names]]
# Also remove polyclonal antibodies? gene names that end in Rik 
adata = adata[:, [not x.endswith('Rik') for x in adata.var_names]]

In [ ]:
sc.pl.highest_expr_genes(adata, n_top=20, )

In [ ]:
# Start with very basic filtering 
print('Started with: \n', adata)
print_separator()
sc.pp.filter_cells(adata, min_genes=200) # remove cells that have less than 200 genes expressed
sc.pp.filter_genes(adata, min_cells=5) # remove genes that are expressed in less than 3 cells
print('Finished with: \n', adata)

In [ ]:
# With pp.calculate_qc_metrics, we can compute many metrics very efficiently.
sc.pp.calculate_qc_metrics(adata, percent_top=None, log1p=False, inplace=True)
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts'],
             jitter=0.4, multi_panel=True)

# n_genes_by_counts represents the number of genes that are detected (expressed) in a particular cell after applying a count threshold. 
# total_counts represents the total number of RNA molecules (total count) detected in a given cell.

# For example, cells with extremely low total_counts might be low-quality cells, and cells with very high total_counts 
# might be doublets (two cells that were sequenced together, leading to inflated transcript counts). 

In [ ]:
print("Remove cells with very high values for total_counts, possible doublets")

print('Started with: \n', adata)
print_separator()
adata = adata[adata.obs.total_counts < 0.5e7, :]
print('Finished with: \n', adata)

sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts'],
             jitter=0.4, multi_panel=True)
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts')

#### Total-count normalize (library-size correct) the data matrix to 10,000 reads per cell, so that counts become comparable among cells.

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)

#sc.pl.highly_variable_genes(adata)

adata.raw = adata
adata = adata[:, adata.var.highly_variable]

#Regress out effects of total counts per cell and the percentage of mitochondrial genes expressed. Scale the data to unit variance.
sc.pp.regress_out(adata, ['total_counts'])

#Scale each gene to unit variance. Clip values exceeding standard deviation 10.
sc.pp.scale(adata, max_value=10)

### PCA 

In [ ]:
print("The number of cells and genes going into PCA is: " + str(adata.shape))

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')

In [ ]:
sc.pl.pca(adata, color='cell_ontology_class')

In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)

In [ ]:
sc.tl.umap(adata)

In [ ]:
sc.tl.leiden(adata)

In [ ]:
sc.pl.umap(adata, color=['leiden', 'cell_ontology_class', 'mouse.id'])

In [ ]:
adata

In [ ]:
low_dim_rep = adata.obsm['X_pca']
# add cell type labels to the low dimensional representation
low_dim_rep = pd.DataFrame(low_dim_rep)
# add PCA to each column name 
low_dim_rep.columns = ['PCA_' + str(x) for x in low_dim_rep.columns]
low_dim_rep['cell_id'] = adata.obs['cell_id'].values
low_dim_rep['cell_type'] = adata.obs['cell_ontology_class'].values

# Do the same for X_umap 
low_dim_rep_umap = adata.obsm['X_umap']
low_dim_rep_umap = pd.DataFrame(low_dim_rep_umap)
low_dim_rep_umap.columns = ['UMAP_' + str(x) for x in low_dim_rep_umap.columns]
low_dim_rep_umap['cell_id'] = adata.obs['cell_id'].values
low_dim_rep_umap['cell_type'] = adata.obs['cell_ontology_class'].values

In [ ]:
# save low dimensional embedding for downstream analysis and Psix input...
output_dir = '/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaMurisBrain'

# PCA embedding
output_file = os.path.join(output_dir, 'scanpy_pca_embedding.csv')
low_dim_rep.to_csv(output_file, index=True, header=True)
print('Saved PCA low dimensional embedding to {}'.format(output_file))

# UMAP embedding
output_file = os.path.join(output_dir, 'scanpy_umap_embedding.csv')
low_dim_rep_umap.to_csv(output_file, index=True, header=True)
print('Saved UMAP low dimensional embedding to {}'.format(output_file))

In [ ]:
sc.tl.rank_genes_groups(adata, 'cell_ontology_class', method='t-test')
sc.pl.rank_genes_groups(adata, n_genes=25, sharey=False)